# Zero-Shot Emotion Analysis on Custom Dataset

This notebook demonstrates how to perform **zero-shot emotion detection** using the pretrained RoBERTa model (`pysentimiento/robertuito-emotion-analysis`) on a custom dataset. 

## Step 1 — Imports

We import all necessary libraries:

- `transformers` → For the RoBERTa tokenizer and model  
- `torch` → PyTorch for running the model and tensor operations  
- `pandas` → Dataset loading and manipulation  
- `numpy` → Array operations and metrics  
- `sklearn.model_selection.train_test_split` → Dataset splitting

In [ ]:
# Import necessary libraries
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

## Step 2 — Load Dataset

- Load the CSV dataset.  
- Keep only the `text` column and all emotion label columns.  
- Inspect first few rows for sanity.

In [ ]:
# Load your CSV dataset
df = pd.read_csv("goemotions_cleaned.csv")  

# Keep only text and emotion columns
emotion_cols = df.columns[9:]  # assuming first column is 'text'
df = df[["text"] + list(emotion_cols)]

print(f"Dataset loaded: {len(df)} rows")
df.head()

## Step 3 — Split Dataset

- Goal: Split dataset into **train / validation / test** sets
    - Train: 70% (for model training/fine-tuning)
    - Validation: 15% (for tuning thresholds/hyperparameters)
    - Test: 15% (for final evaluation)
- `random_state` ensures reproducibility
- `shuffle=True` randomizes data before splitting

In [ ]:
# Split dataset into train, validation, and test sets
df_train, df_temp = train_test_split(df, test_size=0.3, random_state=42, shuffle=True) #splits df_train to 70%, df_temp to 30%
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42, shuffle=True) #splits df_temp into df_val and df_test into 15% respectively

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

## Step 4 — Load Model and Tokenizer

- Load pretrained RoBERTa model for emotion classification.  
- Load the corresponding tokenizer to convert text to model input.  
- `labels` maps model output indices to emotion names.

In [ ]:
# Load pretrained RoBERTa model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("pysentimiento/robertuito-emotion-analysis")
model = AutoModelForSequenceClassification.from_pretrained("pysentimiento/robertuito-emotion-analysis")

# Get mapping from class index to emotion label
# Get emotion labels from the model configuration (print(labels) to see the different labels)

labels = model.config.id2label
print(labels)

## Step 5 — Single-Text Sanity Check

- Check the model's predictions on a few examples from the **test set**.  
- Steps:  
    - Tokenize the text  
    - Run through the model to get logits  
    - Convert logits to probabilities using softmax  
    - Pick **top 3 emotions** and print probabilities

In [ ]:
# Optional: test a few examples from test set
texts = df_test["text"].tolist()  # convert test texts to a list

for t in texts[0:5]:  # loop through first 5 texts
    inputs = tokenizer(t, return_tensors="pt")  # tokenize text for model input
    outputs = model(**inputs)  # feeds the text into the model, and get model predictions
    probs = torch.softmax(outputs.logits, dim=1)  # after the model processes your output, it output logits (raw numbers) that show how strongly the model feels about that emotion. Then, convert logits to probabilities
    
    top3 = torch.topk(probs, k=3)  # get top 3 predicted emotions
    top3_probs = top3.values[0]    # probabilities of top 3
    top3_indices = top3.indices[0] # indices of top 3
    top3_emotions = [labels[i.item()] for i in top3_indices]  # map indices to emotion labels
    
    print(f"Text: {t}")
    for i in range(3):
        print(f"  {i+1}. {top3_emotions[i]} ({top3_probs[i].item()*100:.2f}%)")  # show top 3 emotions
    print("\n")

## Step 6 — Batch Inference on Test Set

- Process the test set in batches (default batch size = 32) for efficiency.  
- Steps:  
    - Tokenize each batch with padding and truncation  
    - Run the model to get logits  
    - Apply softmax to get probabilities  
    - Store probabilities in a NumPy array for later evaluation

To-do: 
- find an english version of Robert E.g. roberta-base (general-purpose, Hugging Face)
- after splitting the data, fine-tune the model using huggingface
- evaluation of results